# Build Origin Shot Sequences

Purpose: build one row per origin shot decision / shot-created flurry.

This notebook starts from `shot_value_base.parquet`, which has one row per evaluated chance with origin context.

The key denominator changes here:

- `03_build_shot_value_dataset.ipynb`: one row per evaluated chance
- `04_build_origin_shot_sequences.ipynb`: one row per origin shot decision / flurry

A deflection is still an evaluated chance, but in this notebook it is attached downstream of the origin shot decision. Rebounds and follow-up chances are also attached to the origin sequence rather than treated as independent shot-decision denominators.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# Resolve project paths whether launched from project root or notebooks folder.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data exists:", DATA_RAW.exists())
print("Processed data exists:", DATA_PROCESSED.exists())

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
Raw data exists: True
Processed data exists: True


## 2. Load Inputs

This notebook uses:

- `shot_value_base.parquet`: evaluated chances with origin context
- `events.parquet`: raw event table, used only for sequence validation and goal rows

Chain construction should be based on evaluated chances, not raw event rows. Raw assists, passes, receptions, and pressure events are not used to extend the chance chain because their event timing can reflect event grammar rather than causal shot sequencing.

In [3]:
shot_value_base = pd.read_parquet(DATA_PROCESSED / "shot_value_base.parquet")
events = pd.read_parquet(DATA_RAW / "events.parquet")

events["period_time"] = pd.to_numeric(events["period_time"], errors="coerce")

print("shot_value_base:", shot_value_base.shape)
print("events:", events.shape)

shot_value_base: (53980, 48)
events: (1800464, 24)


In [4]:
shot_value_base.head()

,game_id,period,chance_period_time,sequence_id,chance_game_stint,chance_event_id,event_type,outcome,description,detail,...,origin_team,origin_team_id,origin_opp_team,origin_opp_team_id,origin_detail,origin_description,chance_has_tracking_data,chance_event_player_tracked,origin_has_tracking_data,origin_event_player_tracked
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,125.27,2,15.0,102,shot,successful,SLOT SHOT FOR ONNET,slot,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,slot,SLOT SHOT FOR ONNET,1,1,1.0,1.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,142.20,2,15.0,111,shot,failed,OUTSIDE SHOT FOR MISSED,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR MISSED,1,1,1.0,1.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,155.27,2,15.0,122,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR ONNET,1,0,1.0,0.0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,161.83,2,15.0,129,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR ONNET,1,1,1.0,1.0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,174.97,2,16.0,138,shot,failed,OUTSIDE SHOT FOR BLOCKED,outsideblocked,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outsideblocked,OUTSIDE SHOT FOR BLOCKED,1,0,1.0,0.0


In [5]:
shot_value_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 53980 entries, 0 to 53979
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   game_id                      53980 non-null  str    
 1   period                       53980 non-null  int64  
 2   chance_period_time           53980 non-null  float64
 3   sequence_id                  53980 non-null  int64  
 4   chance_game_stint            53961 non-null  float64
 5   chance_event_id              53980 non-null  int64  
 6   event_type                   53980 non-null  str    
 7   outcome                      53980 non-null  str    
 8   description                  53980 non-null  str    
 9   detail                       53980 non-null  str    
 10  player_id                    53980 non-null  str    
 11  player_name                  53980 non-null  str    
 12  team                         53980 non-null  str    
 13  team_id                    

## 3. Validate Input Table

Before building shot-created chains, we validate the base chance table.

Expected structure:

- one row per evaluated chance
- unique `game_id + chance_event_id`
- normal shots have `origin_event_id == chance_event_id`
- deflections usually have `origin_event_id != chance_event_id`
- only the two invalid long-delay deflections should have missing origin fields

In [6]:
# One row per evaluated chance event.

shot_value_base.duplicated(["game_id", "chance_event_id"]).sum()

np.int64(0)

In [7]:
shot_value_base["event_type"].value_counts(dropna=False)

event_type
shot          51933
deflection     2047
Name: count, dtype: int64

In [8]:
# Missing origin fields should be limited to the two invalid deflection links.

shot_value_base[
    [
        "origin_event_id",
        "origin_shooter_id",
        "origin_shooter_name",
        "origin_team",
        "origin_team_id",
    ]
].isna().sum()

origin_event_id        2
origin_shooter_id      2
origin_shooter_name    2
origin_team            2
origin_team_id         2
dtype: int64

In [9]:
# Normal shots should be their own origin event.

normal_shots = shot_value_base[shot_value_base["event_type"] == "shot"].copy()

(normal_shots["origin_event_id"] == normal_shots["chance_event_id"]).value_counts(dropna=False)

True    51933
Name: count, dtype: int64

In [10]:
# Deflections with valid links should have a different origin event from chance event.

deflections = shot_value_base[shot_value_base["event_type"] == "deflection"].copy()

(
    deflections["origin_event_id"].notna()
    & (deflections["origin_event_id"] != deflections["chance_event_id"])
).value_counts(dropna=False)

True     2045
False       2
Name: count, dtype: int64

## 4. Validate Sequence Assumptions

The chain boundary will be:

- same `game_id`
- same `period`
- same `sequence_id`
- same attacking team
- rolling time window between same-team evaluated chances

We do not cross `sequence_id`.

We do not use opponent events as hard stops.

We do not use raw assists or goal rows to extend the chain.

In [11]:
# Check whether sequence_id is scoped within games and periods.

sequence_period_counts = (
    events
    .groupby(["game_id", "sequence_id"])["period"]
    .nunique()
    .value_counts()
    .sort_index()
)

sequence_period_counts

period
1    28262
Name: count, dtype: int64

In [12]:
# Event IDs should be unique within game.
# They may not be globally unique across the full dataset.

event_id_uniqueness = {
    "duplicated_sl_event_id_global": events.duplicated(["sl_event_id"]).sum(),
    "duplicated_game_event_id": events.duplicated(["game_id", "sl_event_id"]).sum(),
    "duplicated_game_sequence_event_id": events.duplicated(["game_id", "sequence_id", "sl_event_id"]).sum(),
}

event_id_uniqueness

{'duplicated_sl_event_id_global': np.int64(1796107),
 'duplicated_game_event_id': np.int64(0),
 'duplicated_game_sequence_event_id': np.int64(0)}

In [13]:
# Check whether faceoffs appear more than once inside a game/sequence.
# A faceoff-to-whistle sequence should usually have faceoff activity at the beginning,
# not arbitrary faceoffs later in the same sequence.

faceoff_events = events[events["event_type"] == "faceoff"].copy()

faceoffs_per_sequence = (
    faceoff_events
    .groupby(["game_id", "period", "sequence_id"])
    .agg(
        n_faceoff_rows=("sl_event_id", "count"),
        min_faceoff_time=("period_time", "min"),
        max_faceoff_time=("period_time", "max"),
    )
    .reset_index()
)

faceoffs_per_sequence["faceoff_time_span"] = (
    faceoffs_per_sequence["max_faceoff_time"] - faceoffs_per_sequence["min_faceoff_time"]
)

faceoffs_per_sequence.sort_values("faceoff_time_span", ascending=False).head(20)

,game_id,period,sequence_id,n_faceoff_rows,min_faceoff_time,max_faceoff_time,faceoff_time_span
16145,9c5f5aa5-33ac-51ab-c778-60d24bfacab5,3,71,3,960.03,961.03,1.0
28175,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,2,24,3,255.03,255.03,0.0
30,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,31,3,826.03,826.03,0.0
28191,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,40,3,565.03,565.03,0.0
28190,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,39,3,433.03,433.03,0.0
15,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,16,3,1014.03,1014.03,0.0
28206,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,55,3,1180.03,1180.03,0.0
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,3,0.00,0.00,0.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,3,101.03,101.03,0.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3,3,212.03,212.03,0.0


In [14]:
faceoffs_per_sequence["faceoff_time_span"].describe()

count    28207.000000
mean         0.000035
std          0.005954
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: faceoff_time_span, dtype: float64

## 5. Define Chain Parameters

The shot-created chain uses a rolling follow-up window.

Rule:

1. Start with an origin shot decision.
2. Find the next same-team evaluated chance in the same game, period, and sequence.
3. Include it if it occurs within `FOLLOWUP_WINDOW_SECONDS` of the last included same-team chance.
4. Reset the clock to that included chance.
5. Repeat until no same-team evaluated chance occurs inside the rolling window.

Opponent events do not break the chain, but they also do not reset the clock.

Raw assists, passes, receptions, loose-puck recoveries, and pressure events do not extend the chain.

In [15]:
FOLLOWUP_WINDOW_SECONDS = 5.0
GOAL_BUFFER_SECONDS = 2.0

print("FOLLOWUP_WINDOW_SECONDS:", FOLLOWUP_WINDOW_SECONDS)
print("GOAL_BUFFER_SECONDS:", GOAL_BUFFER_SECONDS)

FOLLOWUP_WINDOW_SECONDS: 5.0
GOAL_BUFFER_SECONDS: 2.0


## 6. Prepare Evaluated Chances For Chain Construction

Chains are built from evaluated chances only.

The ordering key is `chance_period_time`.

`chance_event_id` is used only as a tie-breaker, not as the primary definition of time.

In [16]:
chances = shot_value_base.copy()

# Ensure numeric time fields.
chances["chance_period_time"] = pd.to_numeric(chances["chance_period_time"], errors="coerce")
chances["origin_period_time"] = pd.to_numeric(chances["origin_period_time"], errors="coerce")

# Make sure event IDs are comparable for sorting.
chances["chance_event_id"] = pd.to_numeric(chances["chance_event_id"], errors="coerce")
chances["origin_event_id"] = pd.to_numeric(chances["origin_event_id"], errors="coerce")

# Preserve the raw event-team field, then create the analytical attacking-team field.
#
# Defensive deflections can be credited to the defending player/team even though
# the attacking shot decision belongs to the origin shot team. For chain building,
# those deflections must be grouped with the origin/attacking team, not the raw
# defensive deflection event team.
chances["raw_chance_team_id"] = chances["team_id"]
chances["analytical_team_id"] = chances["team_id"]

is_valid_deflection_with_origin_team = (
    (chances["event_type"] == "deflection")
    & chances["origin_team_id"].notna()
)

chances.loc[is_valid_deflection_with_origin_team, "analytical_team_id"] = chances.loc[
    is_valid_deflection_with_origin_team,
    "origin_team_id",
]

chances["team_id_was_overridden_for_chain"] = (
    chances["raw_chance_team_id"] != chances["analytical_team_id"]
)

# Stable sort for chance-chain construction.
chances = chances.sort_values(
    [
        "game_id",
        "period",
        "sequence_id",
        "analytical_team_id",
        "chance_period_time",
        "chance_event_id",
    ]
).reset_index(drop=True)

chances.head()


,game_id,period,chance_period_time,sequence_id,chance_game_stint,chance_event_id,event_type,outcome,description,detail,...,origin_opp_team_id,origin_detail,origin_description,chance_has_tracking_data,chance_event_player_tracked,origin_has_tracking_data,origin_event_player_tracked,raw_chance_team_id,analytical_team_id,team_id_was_overridden_for_chain
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,190.00,2,23.0,151,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,outside,OUTSIDE SHOT FOR ONNET,1,1,1.0,1.0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,False
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,211.37,2,25.0,174,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,outside,OUTSIDE SHOT FOR ONNET,1,1,1.0,1.0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,False
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,125.27,2,15.0,102,shot,successful,SLOT SHOT FOR ONNET,slot,...,6cac12e2-0546-2c1a-689f-ab26d8a6355a,slot,SLOT SHOT FOR ONNET,1,1,1.0,1.0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,142.20,2,15.0,111,shot,failed,OUTSIDE SHOT FOR MISSED,outside,...,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR MISSED,1,1,1.0,1.0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,155.27,2,15.0,122,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR ONNET,1,0,1.0,0.0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False


In [17]:
# Check missing times. These would break rolling-window logic.

chances[
    [
        "chance_period_time",
        "origin_period_time",
        "chance_event_id",
        "origin_event_id",
    ]
].isna().sum()

chance_period_time    0
origin_period_time    2
chance_event_id       0
origin_event_id       2
dtype: int64

In [18]:
shot_value_base.duplicated(["game_id", "chance_event_id"]).sum()
shot_value_base["event_type"].value_counts(dropna=False)
shot_value_base[["origin_event_id","origin_shooter_id","origin_shooter_name","origin_team","origin_team_id"]].isna().sum()
(normal_shots["origin_event_id"] == normal_shots["chance_event_id"]).value_counts(dropna=False)
(deflections["origin_event_id"].notna() & (deflections["origin_event_id"] != deflections["chance_event_id"])).value_counts(dropna=False)
sequence_period_counts
event_id_uniqueness
faceoffs_per_sequence["faceoff_time_span"].describe()
chances[["chance_period_time","origin_period_time","chance_event_id","origin_event_id"]].isna().sum()

chance_period_time    0
origin_period_time    2
chance_event_id       0
origin_event_id       2
dtype: int64

## 7. Build Non-Overlapping Rolling Chance Chains

The denominator is now a shot-created chain, not every evaluated chance.

Within each game, period, sequence, and team:

1. Sort evaluated chances by `chance_period_time`.
2. Start a chain with the first unassigned chance.
3. Include later same-team evaluated chances if they occur within `FOLLOWUP_WINDOW_SECONDS` of the last included chance.
4. Each included chance resets the clock.
5. Mark included chances as assigned so they cannot start another chain.

Opponent events are not used as hard stops, but only same-team evaluated chances reset the rolling clock.

This avoids overlapping chains where a rebound is both part of the original shot's chain and also treated as a separate new origin chain.

In [19]:
# Chain construction uses evaluated chances with valid timing.
# We keep invalid origin deflection rows as evaluated chances, but they should not
# define reliable origin-shot context later.
#
# Chains are grouped by analytical_team_id, not raw team_id. This matters for
# defensive deflection rows, where the event can be credited to the defending team
# even though the attacking chance chain belongs to the origin shot team.

chain_input = chances.copy()

chain_input = chain_input.sort_values(
    [
        "game_id",
        "period",
        "sequence_id",
        "analytical_team_id",
        "chance_period_time",
        "chance_event_id",
    ]
).reset_index(drop=True)

{
    "rows": len(chain_input),
    "team_overrides_for_chain": int(chain_input["team_id_was_overridden_for_chain"].sum()),
}


{'rows': 53980, 'team_overrides_for_chain': 5}

In [20]:
# Inspect chance rows whose raw event team is overridden for chain construction.
# Expected: rare defensive deflection rows.

chain_input.loc[
    chain_input["team_id_was_overridden_for_chain"],
    [
        "game_id",
        "period",
        "sequence_id",
        "chance_event_id",
        "chance_period_time",
        "event_type",
        "raw_chance_team_id",
        "analytical_team_id",
        "origin_team_id",
        "origin_event_id",
        "origin_location",
        "chance_location",
        "sl_xg_all_shots",
    ],
].head(20)


,game_id,period,sequence_id,chance_event_id,chance_period_time,event_type,raw_chance_team_id,analytical_team_id,origin_team_id,origin_event_id,origin_location,chance_location,sl_xg_all_shots
10724,3cf0528d-a562-f62d-5560-93079528b9c7,3,52,2592,160.37,deflection,94373660-74b2-7b92-acca-ac87ea867b06,08dc0c28-d913-12fb-f899-7b366b3a2f65,08dc0c28-d913-12fb-f899-7b366b3a2f65,2590.0,outside,slot,0.711449
14326,4bac62e1-2e7d-4df0-2bcf-b12f5b30a7ce,3,48,3744,1078.43,deflection,f630ab4e-2bc7-6672-9c9c-4a4c835b198c,dceeb728-b0b4-4d97-b909-69aca4243d90,dceeb728-b0b4-4d97-b909-69aca4243d90,3741.0,outside,slot,0.295925
21942,6c660c51-6fbf-aec3-1671-02e0fde52ce6,3,45,3296,822.13,deflection,2251a5f6-9710-c610-8d44-9f2e4f4fc7a2,be8d53f2-8cbd-df04-b9e7-0eb07a851c0a,be8d53f2-8cbd-df04-b9e7-0eb07a851c0a,3294.0,outside,slot,0.707359
31626,a15753d9-b807-9e24-d9f6-85fa54fc426a,1,10,702,697.63,deflection,8dbadcf1-c49b-6e64-a846-b145c2c5a426,9c9ce715-dcef-1692-1850-eba56333387b,9c9ce715-dcef-1692-1850-eba56333387b,698.0,slot,slot,0.274347
51421,f5966889-a00b-ffcd-6bef-c3bcfa17bfe2,3,47,2600,228.27,deflection,19188913-73c8-34a7-d02a-5d1c3e58a191,fbd255f8-427d-f29b-e83e-6c631c75899a,fbd255f8-427d-f29b-e83e-6c631c75899a,2596.0,outside,slot,0.356813


In [21]:
def build_non_overlapping_chains_for_group(group, followup_window_seconds):
    """Build non-overlapping rolling chance chains within one game/period/sequence/team group.

    The group contains same-team evaluated chances in one whistle-to-whistle sequence.
    Team means analytical attacking team, not necessarily the raw event-team field.

    A chance can belong to only one chain. Once included, it is marked assigned and
    cannot later start another chain.
    """
    group = group.sort_values(["chance_period_time", "chance_event_id"]).reset_index(drop=True)

    assigned = np.zeros(len(group), dtype=bool)
    chain_rows = []
    chain_id_local = 0

    for start_idx in range(len(group)):
        if assigned[start_idx]:
            continue

        chain_id_local += 1

        start_row = group.iloc[start_idx]
        last_included_time = start_row["chance_period_time"]

        # Start a new chain with the first unassigned evaluated chance.
        assigned[start_idx] = True

        chain_rows.append(
            {
                "chain_id_local": chain_id_local,
                "chain_order": 1,
                "game_id": start_row["game_id"],
                "period": start_row["period"],
                "sequence_id": start_row["sequence_id"],
                "team_id": start_row["analytical_team_id"],
                "raw_chance_team_id": start_row["raw_chance_team_id"],
                "team_id_was_overridden_for_chain": start_row["team_id_was_overridden_for_chain"],
                "chance_event_id": start_row["chance_event_id"],
                "chance_period_time": start_row["chance_period_time"],
                "seconds_since_last_included_chance": 0.0,
                "event_type": start_row["event_type"],
                "chance_location": start_row["chance_location"],
                "origin_event_id": start_row["origin_event_id"],
                "origin_period_time": start_row["origin_period_time"],
                "origin_location": start_row["origin_location"],
                "sl_xg_all_shots": start_row["sl_xg_all_shots"],
                "goal_within_2s_same_team": start_row["goal_within_2s_same_team"],
            }
        )

        chain_order = 1

        # Include later unassigned chances inside the rolling window.
        for next_idx in range(start_idx + 1, len(group)):
            if assigned[next_idx]:
                continue

            next_row = group.iloc[next_idx]
            seconds_since_last = next_row["chance_period_time"] - last_included_time

            if seconds_since_last <= followup_window_seconds:
                assigned[next_idx] = True
                chain_order += 1
                last_included_time = next_row["chance_period_time"]

                chain_rows.append(
                    {
                        "chain_id_local": chain_id_local,
                        "chain_order": chain_order,
                        "game_id": next_row["game_id"],
                        "period": next_row["period"],
                        "sequence_id": next_row["sequence_id"],
                        "team_id": next_row["analytical_team_id"],
                        "raw_chance_team_id": next_row["raw_chance_team_id"],
                        "team_id_was_overridden_for_chain": next_row["team_id_was_overridden_for_chain"],
                        "chance_event_id": next_row["chance_event_id"],
                        "chance_period_time": next_row["chance_period_time"],
                        "seconds_since_last_included_chance": seconds_since_last,
                        "event_type": next_row["event_type"],
                        "chance_location": next_row["chance_location"],
                        "origin_event_id": next_row["origin_event_id"],
                        "origin_period_time": next_row["origin_period_time"],
                        "origin_location": next_row["origin_location"],
                        "sl_xg_all_shots": next_row["sl_xg_all_shots"],
                        "goal_within_2s_same_team": next_row["goal_within_2s_same_team"],
                    }
                )
            else:
                # Because the group is time-sorted, later chances will be even farther away
                # unless the clock is reset. Since this candidate was not included, the
                # current chain ends.
                break

    return pd.DataFrame(chain_rows)


In [22]:
# Build non-overlapping chain membership across all game/period/sequence/analytical-team groups.

group_keys = ["game_id", "period", "sequence_id", "analytical_team_id"]

chain_parts = []

for _, group in chain_input.groupby(group_keys, sort=False):
    chain_parts.append(
        build_non_overlapping_chains_for_group(
            group=group,
            followup_window_seconds=FOLLOWUP_WINDOW_SECONDS,
        )
    )

chain_membership = pd.concat(chain_parts, ignore_index=True)

chain_membership.shape


(53980, 18)

In [23]:
# Create a globally unique chain ID using game/period/sequence/analytical-team/local chain order.
# Here team_id is the analytical attacking team for the chain.

chain_membership["chain_id"] = (
    chain_membership["game_id"].astype(str)
    + "_p" + chain_membership["period"].astype(str)
    + "_s" + chain_membership["sequence_id"].astype(str)
    + "_t" + chain_membership["team_id"].astype(str)
    + "_c" + chain_membership["chain_id_local"].astype(str)
)

chain_membership.head()


,chain_id_local,chain_order,game_id,period,sequence_id,team_id,raw_chance_team_id,team_id_was_overridden_for_chain,chance_event_id,chance_period_time,seconds_since_last_included_chance,event_type,chance_location,origin_event_id,origin_period_time,origin_location,sl_xg_all_shots,goal_within_2s_same_team,chain_id
0,1,1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,False,151,190.00,0.0,shot,outside,151.0,190.00,outside,0.003101,False,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_t6c...
1,2,1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,False,174,211.37,0.0,shot,outside,174.0,211.37,outside,0.006018,False,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_t6c...
2,1,1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False,102,125.27,0.0,shot,slot,102.0,125.27,slot,0.106327,False,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...
3,2,1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False,111,142.20,0.0,shot,outside,111.0,142.20,outside,0.002135,False,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...
4,3,1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False,122,155.27,0.0,shot,slot,122.0,155.27,slot,0.018751,False,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...


In [24]:
# Every evaluated chance should appear exactly once in chain_membership.

chain_membership.duplicated(["game_id", "chance_event_id"]).sum()

np.int64(0)

In [25]:
# Row count should match the evaluated chance table exactly.

{
    "chain_membership_rows": len(chain_membership),
    "shot_value_base_rows": len(shot_value_base),
    "difference": len(chain_membership) - len(shot_value_base),
}

{'chain_membership_rows': 53980,
 'shot_value_base_rows': 53980,
 'difference': 0}

In [26]:
# Chain size distribution.

chain_counts = (
    chain_membership
    .groupby("chain_id")
    .size()
    .rename("n_chances_in_chain")
    .reset_index()
)

chain_counts["n_chances_in_chain"].value_counts().sort_index()

n_chances_in_chain
1    43960
2     4188
3      466
4       51
5        7
7        1
Name: count, dtype: int64

In [27]:
# Chain duration diagnostics.

chain_time_summary = (
    chain_membership
    .groupby("chain_id")
    .agg(
        chain_start_time=("chance_period_time", "min"),
        chain_end_time=("chance_period_time", "max"),
        n_chances_in_chain=("chance_event_id", "count"),
    )
    .reset_index()
)

chain_time_summary["chain_duration_seconds"] = (
    chain_time_summary["chain_end_time"] - chain_time_summary["chain_start_time"]
)

chain_time_summary["chain_duration_seconds"].describe()

count    48673.000000
mean         0.269161
std          0.969809
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         17.600000
Name: chain_duration_seconds, dtype: float64

In [28]:
# Longest chains for inspection.

chain_time_summary.sort_values(
    ["chain_duration_seconds", "n_chances_in_chain"],
    ascending=False,
).head(20)

,chain_id,chain_start_time,chain_end_time,n_chances_in_chain,chain_duration_seconds
16356,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...,652.00,669.60,5,17.60
20907,72e5d0ee-3e65-a722-56a1-a5d8d48bcd2a_p3_s66_t6...,759.93,772.40,4,12.47
14908,55864f17-89ae-90fc-a24e-5fb25f48aad0_p3_s50_t2...,431.57,443.13,4,11.56
22108,7b1be823-d347-60cf-e6db-3a13e78f91c7_p3_s53_t8...,874.13,885.63,4,11.50
7621,301551ea-a94e-5ad0-bd89-8aa1a3c5b198_p3_s39_t8...,64.53,75.37,4,10.84
41441,dd89ead8-402e-8592-d9c3-4799b30b7bae_p3_s58_te...,961.73,972.37,4,10.64
31585,b55785a0-7bd8-584e-ec20-1542510ce392_p1_s18_t6...,1175.20,1185.80,5,10.60
26273,959fd163-6ca0-9edd-d34b-ddd075bddc6d_p2_s26_tb...,516.87,527.20,4,10.33
12720,4ad5f0f8-a841-eee7-5322-91ec44e1e9bf_p2_s25_t8...,155.83,166.07,4,10.24
24507,8a5c5406-a2ca-05a0-bda4-22bc865cbb44_p3_s49_t2...,784.57,794.77,4,10.20


In [29]:
# Inspect the longest chain's member chances.

longest_chain_id = chain_time_summary.sort_values(
    "chain_duration_seconds",
    ascending=False,
).iloc[0]["chain_id"]

chain_membership[
    chain_membership["chain_id"] == longest_chain_id
].sort_values("chain_order")

,chain_id_local,chain_order,game_id,period,sequence_id,team_id,raw_chance_team_id,team_id_was_overridden_for_chain,chance_event_id,chance_period_time,seconds_since_last_included_chance,event_type,chance_location,origin_event_id,origin_period_time,origin_location,sl_xg_all_shots,goal_within_2s_same_team,chain_id
18150,2,1,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,False,751,652.00,0.00,shot,slot,751.0,652.00,slot,0.023213,False,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...
18151,2,2,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,False,758,656.60,4.60,shot,outside,758.0,656.60,outside,0.005574,False,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...
18152,2,3,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,False,765,660.97,4.37,shot,outside,765.0,660.97,outside,0.002636,False,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...
18153,2,4,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,False,768,665.10,4.13,shot,outside,768.0,665.10,outside,0.001855,False,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...
18154,2,5,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,False,775,669.60,4.50,shot,outside,775.0,669.60,outside,0.140342,False,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...


## 8. Aggregate Chain-Level Features

Each `chain_id` now represents one non-overlapping same-team shot-created chain.

This section collapses chain membership into one row per chain.

The first evaluated chance in the chain defines the chain anchor. For a normal shot, the anchor is the shot itself. For a deflection, the anchor preserves the linked origin shot event from `03_build_shot_value_dataset.ipynb`.

In [30]:
# First chance in each chain defines the chain anchor.

chain_starts = (
    chain_membership
    .sort_values(["chain_id", "chain_order"])
    .groupby("chain_id", as_index=False)
    .first()
)

chain_starts.shape

(48673, 19)

In [31]:
# Confirm one start row per chain.

{
    "n_chain_starts": len(chain_starts),
    "n_unique_chains": chain_membership["chain_id"].nunique(),
    "difference": len(chain_starts) - chain_membership["chain_id"].nunique(),
}

{'n_chain_starts': 48673, 'n_unique_chains': 48673, 'difference': 0}

In [32]:
# Aggregate chance-level information to one row per chain.

chain_agg = (
    chain_membership
    .groupby("chain_id")
    .agg(
        game_id=("game_id", "first"),
        period=("period", "first"),
        sequence_id=("sequence_id", "first"),
        team_id=("team_id", "first"),
        raw_first_chance_team_id=("raw_chance_team_id", "first"),
        n_team_id_overrides_for_chain=("team_id_was_overridden_for_chain", "sum"),
        n_chances_in_chain=("chance_event_id", "count"),
        chain_start_time=("chance_period_time", "min"),
        chain_end_time=("chance_period_time", "max"),
        first_evaluated_chance_event_id=("chance_event_id", "first"),
        first_evaluated_chance_time=("chance_period_time", "first"),
        first_evaluated_event_type=("event_type", "first"),
        first_evaluated_location=("chance_location", "first"),
        first_evaluated_xg=("sl_xg_all_shots", "first"),
        chain_max_xg=("sl_xg_all_shots", "max"),
        chain_sum_xg=("sl_xg_all_shots", "sum"),
        chain_any_goal_within_2s_flag=("goal_within_2s_same_team", "max"),
    )
    .reset_index()
)

chain_agg["chain_duration_seconds"] = (
    chain_agg["chain_end_time"] - chain_agg["chain_start_time"]
)

chain_agg.head()


,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,n_chances_in_chain,chain_start_time,chain_end_time,first_evaluated_chance_event_id,first_evaluated_chance_time,first_evaluated_event_type,first_evaluated_location,first_evaluated_xg,chain_max_xg,chain_sum_xg,chain_any_goal_within_2s_flag,chain_duration_seconds
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,1,591.73,591.73,565,591.73,shot,outside,0.040997,0.040997,0.040997,False,0.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,722.40,722.40,687,722.40,shot,outside,0.015540,0.015540,0.015540,False,0.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,736.33,736.33,702,736.33,shot,slot,0.010975,0.010975,0.010975,False,0.0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,765.73,765.73,730,765.73,shot,slot,0.089858,0.089858,0.089858,False,0.0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,923.80,923.80,882,923.80,shot,outside,0.010167,0.010167,0.010167,False,0.0


In [33]:
# Add chain anchor/origin fields from the first included chance.

chain_anchor_columns = [
    "chain_id",
    "origin_event_id",
    "origin_period_time",
    "origin_location",
    "chance_event_id",
    "chance_period_time",
    "event_type",
    "chance_location",
]

chain_anchors = chain_starts[chain_anchor_columns].rename(
    columns={
        "origin_event_id": "anchor_origin_event_id",
        "origin_period_time": "anchor_origin_period_time",
        "origin_location": "anchor_origin_location",
        "chance_event_id": "anchor_first_chance_event_id",
        "chance_period_time": "anchor_first_chance_time",
        "event_type": "anchor_first_event_type",
        "chance_location": "anchor_first_chance_location",
    }
)

origin_shot_sequences = chain_agg.merge(
    chain_anchors,
    how="left",
    on="chain_id",
)

origin_shot_sequences.shape

(48673, 26)

## 9. Add Deflection and Follow-Up Features

Deflections are evaluated chances downstream of an origin shot decision.

Follow-up chances are evaluated chances after the first chance in the same non-overlapping chain.

The headline value metric should be `chain_max_xg`, not `chain_sum_xg`, because chances in a flurry are not independent lottery tickets.

In [34]:
# Deflection features by chain.

deflection_features = (
    chain_membership[chain_membership["event_type"] == "deflection"]
    .groupby("chain_id")
    .agg(
        has_deflection=("chance_event_id", lambda s: True),
        n_deflections=("chance_event_id", "count"),
        first_deflection_event_id=("chance_event_id", "first"),
        first_deflection_time=("chance_period_time", "first"),
        deflection_max_xg=("sl_xg_all_shots", "max"),
        deflection_sum_xg=("sl_xg_all_shots", "sum"),
    )
    .reset_index()
)

origin_shot_sequences = origin_shot_sequences.merge(
    deflection_features,
    how="left",
    on="chain_id",
)

origin_shot_sequences["has_deflection"] = (
    origin_shot_sequences["has_deflection"].fillna(False)
)
origin_shot_sequences["n_deflections"] = (
    origin_shot_sequences["n_deflections"].fillna(0).astype(int)
)

origin_shot_sequences.head()

,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,n_chances_in_chain,chain_start_time,chain_end_time,...,anchor_first_chance_event_id,anchor_first_chance_time,anchor_first_event_type,anchor_first_chance_location,has_deflection,n_deflections,first_deflection_event_id,first_deflection_time,deflection_max_xg,deflection_sum_xg
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,1,591.73,591.73,...,565,591.73,shot,outside,False,0,NaN,NaN,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,722.40,722.40,...,687,722.40,shot,outside,False,0,NaN,NaN,NaN,NaN
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,736.33,736.33,...,702,736.33,shot,slot,False,0,NaN,NaN,NaN,NaN
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,765.73,765.73,...,730,765.73,shot,slot,False,0,NaN,NaN,NaN,NaN
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,923.80,923.80,...,882,923.80,shot,outside,False,0,NaN,NaN,NaN,NaN


In [35]:
# Follow-up features exclude the first evaluated chance in the chain.

followup_chances = chain_membership[chain_membership["chain_order"] > 1].copy()

followup_features = (
    followup_chances
    .groupby("chain_id")
    .agg(
        has_followup_chance=("chance_event_id", lambda s: True),
        n_followup_chances=("chance_event_id", "count"),
        first_followup_event_id=("chance_event_id", "first"),
        first_followup_time=("chance_period_time", "first"),
        followup_max_xg=("sl_xg_all_shots", "max"),
        followup_sum_xg=("sl_xg_all_shots", "sum"),
    )
    .reset_index()
)

origin_shot_sequences = origin_shot_sequences.merge(
    followup_features,
    how="left",
    on="chain_id",
)

origin_shot_sequences["has_followup_chance"] = (
    origin_shot_sequences["has_followup_chance"].fillna(False)
)
origin_shot_sequences["n_followup_chances"] = (
    origin_shot_sequences["n_followup_chances"].fillna(0).astype(int)
)

origin_shot_sequences.head()

,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,n_chances_in_chain,chain_start_time,chain_end_time,...,first_deflection_event_id,first_deflection_time,deflection_max_xg,deflection_sum_xg,has_followup_chance,n_followup_chances,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,1,591.73,591.73,...,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,722.40,722.40,...,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,736.33,736.33,...,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,765.73,765.73,...,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1,923.80,923.80,...,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN


In [36]:
# Fill numeric follow-up/deflection fields where absence means zero.
# Keep event IDs and times as missing when absent.

zero_fill_columns = [
    "deflection_max_xg",
    "deflection_sum_xg",
    "followup_max_xg",
    "followup_sum_xg",
]

origin_shot_sequences[zero_fill_columns] = (
    origin_shot_sequences[zero_fill_columns].fillna(0.0)
)

origin_shot_sequences[
    [
        "has_deflection",
        "n_deflections",
        "has_followup_chance",
        "n_followup_chances",
        "chain_max_xg",
        "followup_max_xg",
    ]
].head()

,has_deflection,n_deflections,has_followup_chance,n_followup_chances,chain_max_xg,followup_max_xg
0,False,0,False,0,0.040997,0.0
1,False,0,False,0,0.015540,0.0
2,False,0,False,0,0.010975,0.0
3,False,0,False,0,0.089858,0.0
4,False,0,False,0,0.010167,0.0


## 10. Add Chain-Level Goal Label

The chance-level `goal_within_2s_same_team` field from the previous notebook is only a local proximity flag.

Here, we compute a chain-level goal label directly from player-level goal rows.

A chain is marked as a goal chain if the same team has a player-level goal row in the same game, period, and sequence between the first chance and `GOAL_BUFFER_SECONDS` after the last included chance.

In [37]:
# Player-level goal rows have player/team information.
# Game-level goal rows with missing player/team are not used here.

goal_rows = events[
    (events["event_type"] == "goal")
    & (events["player_id"].notna())
    & (events["team_id"].notna())
].copy()

goal_rows["period_time"] = pd.to_numeric(goal_rows["period_time"], errors="coerce")

goal_rows = goal_rows[
    [
        "game_id",
        "period",
        "sequence_id",
        "period_time",
        "team_id",
        "player_id",
        "player_name",
        "sl_event_id",
    ]
].rename(
    columns={
        "period_time": "goal_time",
        "team_id": "goal_team_id",
        "player_id": "goal_player_id",
        "player_name": "goal_player_name",
        "sl_event_id": "goal_event_id",
    }
)

goal_rows.shape

(2816, 8)

In [38]:
# Build chain-level goal labels.
# This avoids interpreting the 03 chance-level proximity flag as direct attribution.
#
# team_id in origin_shot_sequences is the analytical attacking team for the chain,
# which fixes defensive-deflection rows credited to the defending team.

chain_goal_rows = []

for _, chain in origin_shot_sequences.iterrows():
    candidates = goal_rows[
        (goal_rows["game_id"] == chain["game_id"])
        & (goal_rows["period"] == chain["period"])
        & (goal_rows["sequence_id"] == chain["sequence_id"])
        & (goal_rows["goal_team_id"] == chain["team_id"])
        & (goal_rows["goal_time"] >= chain["chain_start_time"])
        & (goal_rows["goal_time"] <= chain["chain_end_time"] + GOAL_BUFFER_SECONDS)
    ].sort_values(["goal_time", "goal_event_id"])

    chain_goal_rows.append(
        {
            "chain_id": chain["chain_id"],
            "chain_goal": not candidates.empty,
            "chain_goal_event_id": candidates["goal_event_id"].iloc[0] if not candidates.empty else pd.NA,
            "chain_goal_time": candidates["goal_time"].iloc[0] if not candidates.empty else pd.NA,
            "chain_goal_player_id": candidates["goal_player_id"].iloc[0] if not candidates.empty else pd.NA,
            "chain_goal_player_name": candidates["goal_player_name"].iloc[0] if not candidates.empty else pd.NA,
        }
    )

chain_goal_labels = pd.DataFrame(chain_goal_rows)

origin_shot_sequences = origin_shot_sequences.merge(
    chain_goal_labels,
    how="left",
    on="chain_id",
)

origin_shot_sequences["chain_goal"].value_counts(dropna=False)


chain_goal
False    45906
True      2767
Name: count, dtype: int64

In [39]:
# Reasonableness check: chain goals should not exceed player-level goal rows by a large amount.
# It may be slightly different because of buffer logic, but it should be close.

{
    "chain_goal_count": int(origin_shot_sequences["chain_goal"].sum()),
    "player_level_goal_rows": len(goal_rows),
}

{'chain_goal_count': 2767, 'player_level_goal_rows': 2816}

## 11. Validate Chain-Level Features

Before saving, we validate the chain-level table.

The main checks are:

- one row per chain
- no duplicate chain IDs
- every chain has at least one evaluated chance
- follow-up and deflection counts are plausible
- `chain_max_xg` is at least as large as first-chance xG
- goal labeling is close to player-level goal rows

In [40]:
# One row per chain.

{
    "rows": len(origin_shot_sequences),
    "unique_chain_ids": origin_shot_sequences["chain_id"].nunique(),
    "duplicate_chain_ids": origin_shot_sequences.duplicated("chain_id").sum(),
}

{'rows': 48673, 'unique_chain_ids': 48673, 'duplicate_chain_ids': np.int64(0)}

In [41]:
# Chain size and duration distributions.

origin_shot_sequences[
    [
        "n_chances_in_chain",
        "chain_duration_seconds",
        "first_evaluated_xg",
        "chain_max_xg",
        "chain_sum_xg",
        "n_deflections",
        "n_followup_chances",
        "followup_max_xg",
    ]
].describe()

,n_chances_in_chain,chain_duration_seconds,first_evaluated_xg,chain_max_xg,chain_sum_xg,n_deflections,n_followup_chances,followup_max_xg
count,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000
mean,1.109034,0.269161,0.045373,0.053214,0.057150,0.042056,0.109034,0.010892
std,0.353448,0.969809,0.078502,0.090148,0.101723,0.200924,0.353448,0.054311
min,1.000000,0.000000,0.000610,0.000610,0.000610,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.004025,0.004435,0.004502,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.015797,0.018683,0.018949,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.051575,0.061151,0.063316,0.000000,0.000000,0.000000
max,7.000000,17.600000,0.938368,0.938368,1.462747,2.000000,6.000000,0.854483


In [42]:
# Logical consistency checks.

consistency_checks = {
    "chain_max_less_than_first_xg": (
        origin_shot_sequences["chain_max_xg"]
        < origin_shot_sequences["first_evaluated_xg"]
    ).sum(),
    "followup_true_but_zero_count": (
        origin_shot_sequences["has_followup_chance"]
        & (origin_shot_sequences["n_followup_chances"] == 0)
    ).sum(),
    "followup_false_but_positive_count": (
        (~origin_shot_sequences["has_followup_chance"])
        & (origin_shot_sequences["n_followup_chances"] > 0)
    ).sum(),
    "deflection_true_but_zero_count": (
        origin_shot_sequences["has_deflection"]
        & (origin_shot_sequences["n_deflections"] == 0)
    ).sum(),
    "deflection_false_but_positive_count": (
        (~origin_shot_sequences["has_deflection"])
        & (origin_shot_sequences["n_deflections"] > 0)
    ).sum(),
}

consistency_checks

{'chain_max_less_than_first_xg': np.int64(0),
 'followup_true_but_zero_count': np.int64(0),
 'followup_false_but_positive_count': np.int64(0),
 'deflection_true_but_zero_count': np.int64(0),
 'deflection_false_but_positive_count': np.int64(0)}

In [43]:
# Follow-up and deflection rates.

{
    "chains": len(origin_shot_sequences),
    "followup_chains": int(origin_shot_sequences["has_followup_chance"].sum()),
    "followup_chain_rate": origin_shot_sequences["has_followup_chance"].mean(),
    "deflection_chains": int(origin_shot_sequences["has_deflection"].sum()),
    "deflection_chain_rate": origin_shot_sequences["has_deflection"].mean(),
    "goal_chains": int(origin_shot_sequences["chain_goal"].sum()),
    "goal_chain_rate": origin_shot_sequences["chain_goal"].mean(),
}

{'chains': 48673,
 'followup_chains': 4713,
 'followup_chain_rate': np.float64(0.09682986460666078),
 'deflection_chains': 2045,
 'deflection_chain_rate': np.float64(0.04201508022928523),
 'goal_chains': 2767,
 'goal_chain_rate': np.float64(0.05684876625644608)}

In [44]:
# Key split by anchor origin location.
# This is not the final analysis, but it confirms the table can support the project question.

origin_shot_sequences.groupby("anchor_origin_location", dropna=False).agg(
    chains=("chain_id", "count"),
    mean_first_xg=("first_evaluated_xg", "mean"),
    median_first_xg=("first_evaluated_xg", "median"),
    mean_chain_max_xg=("chain_max_xg", "mean"),
    median_chain_max_xg=("chain_max_xg", "median"),
    followup_rate=("has_followup_chance", "mean"),
    deflection_rate=("has_deflection", "mean"),
    goal_rate=("chain_goal", "mean"),
).reset_index()

,anchor_origin_location,chains,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,followup_rate,deflection_rate,goal_rate
0,outside,30582,0.022323,0.005669,0.029961,0.006495,0.092767,0.059774,0.031751
1,slot,18089,0.084344,0.051446,0.092529,0.054759,0.103709,0.011886,0.099287
2,NaN,2,0.032919,0.032919,0.032919,0.032919,0.0,1.0,0.000000


In [45]:
# Longest chains with key features.

origin_shot_sequences.sort_values(
    ["chain_duration_seconds", "n_chances_in_chain"],
    ascending=False,
).head(20)[
    [
        "chain_id",
        "game_id",
        "period",
        "sequence_id",
        "team_id",
        "raw_first_chance_team_id",
        "n_team_id_overrides_for_chain",
        "chain_start_time",
        "chain_end_time",
        "chain_duration_seconds",
        "n_chances_in_chain",
        "first_evaluated_event_type",
        "anchor_origin_location",
        "first_evaluated_location",
        "first_evaluated_xg",
        "chain_max_xg",
        "chain_goal",
    ]
]


,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,n_chances_in_chain,first_evaluated_event_type,anchor_origin_location,first_evaluated_location,first_evaluated_xg,chain_max_xg,chain_goal
16356,5dda9398-35b9-5c8d-7507-60482037bf84_p1_s15_t2...,5dda9398-35b9-5c8d-7507-60482037bf84,1,15,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,0,652.00,669.60,17.60,5,shot,slot,slot,0.023213,0.140342,False
20907,72e5d0ee-3e65-a722-56a1-a5d8d48bcd2a_p3_s66_t6...,72e5d0ee-3e65-a722-56a1-a5d8d48bcd2a,3,66,67c5ed49-a598-0057-310c-c709186007ab,67c5ed49-a598-0057-310c-c709186007ab,0,759.93,772.40,12.47,4,shot,outside,outside,0.007884,0.035813,False
14908,55864f17-89ae-90fc-a24e-5fb25f48aad0_p3_s50_t2...,55864f17-89ae-90fc-a24e-5fb25f48aad0,3,50,28147617-6a1a-8547-8b07-b84f5d0807c5,28147617-6a1a-8547-8b07-b84f5d0807c5,0,431.57,443.13,11.56,4,shot,outside,outside,0.005944,0.119972,False
22108,7b1be823-d347-60cf-e6db-3a13e78f91c7_p3_s53_t8...,7b1be823-d347-60cf-e6db-3a13e78f91c7,3,53,80e4355b-5866-7f9b-77e7-36b7c804b5ed,80e4355b-5866-7f9b-77e7-36b7c804b5ed,0,874.13,885.63,11.50,4,shot,outside,outside,0.002457,0.037445,False
7621,301551ea-a94e-5ad0-bd89-8aa1a3c5b198_p3_s39_t8...,301551ea-a94e-5ad0-bd89-8aa1a3c5b198,3,39,800ae5b7-9166-d248-f73d-c4d16fd1b4d6,800ae5b7-9166-d248-f73d-c4d16fd1b4d6,0,64.53,75.37,10.84,4,shot,slot,slot,0.042472,0.042472,False
41441,dd89ead8-402e-8592-d9c3-4799b30b7bae_p3_s58_te...,dd89ead8-402e-8592-d9c3-4799b30b7bae,3,58,e3572140-5df5-e208-74d8-cb1a581d5fba,e3572140-5df5-e208-74d8-cb1a581d5fba,0,961.73,972.37,10.64,4,shot,outside,outside,0.002148,0.258610,False
31585,b55785a0-7bd8-584e-ec20-1542510ce392_p1_s18_t6...,b55785a0-7bd8-584e-ec20-1542510ce392,1,18,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,1175.20,1185.80,10.60,5,shot,slot,slot,0.020089,0.207925,False
26273,959fd163-6ca0-9edd-d34b-ddd075bddc6d_p2_s26_tb...,959fd163-6ca0-9edd-d34b-ddd075bddc6d,2,26,be8d53f2-8cbd-df04-b9e7-0eb07a851c0a,be8d53f2-8cbd-df04-b9e7-0eb07a851c0a,0,516.87,527.20,10.33,4,shot,outside,outside,0.003749,0.063386,False
12720,4ad5f0f8-a841-eee7-5322-91ec44e1e9bf_p2_s25_t8...,4ad5f0f8-a841-eee7-5322-91ec44e1e9bf,2,25,80e4355b-5866-7f9b-77e7-36b7c804b5ed,80e4355b-5866-7f9b-77e7-36b7c804b5ed,0,155.83,166.07,10.24,4,shot,outside,outside,0.008905,0.031466,False
24507,8a5c5406-a2ca-05a0-bda4-22bc865cbb44_p3_s49_t2...,8a5c5406-a2ca-05a0-bda4-22bc865cbb44,3,49,28147617-6a1a-8547-8b07-b84f5d0807c5,28147617-6a1a-8547-8b07-b84f5d0807c5,0,784.57,794.77,10.20,4,shot,outside,outside,0.015550,0.049385,False


## 12. Select Final Columns

The saved output has one row per non-overlapping shot-created chain.

This is the denominator used for outside-shot decision analysis.

In [46]:
# Clean final dtypes before saving.
#
# These fields can become object dtype because they are created through merges
# with missing values. Cast them explicitly so downstream notebooks behave
# predictably.

origin_shot_sequences["has_deflection"] = origin_shot_sequences["has_deflection"].astype(bool)
origin_shot_sequences["has_followup_chance"] = origin_shot_sequences["has_followup_chance"].astype(bool)
origin_shot_sequences["chain_goal"] = origin_shot_sequences["chain_goal"].astype(bool)
origin_shot_sequences["chain_any_goal_within_2s_flag"] = origin_shot_sequences["chain_any_goal_within_2s_flag"].astype(bool)

# Keep goal time numeric. Missing goal times remain NaN.
origin_shot_sequences["chain_goal_time"] = pd.to_numeric(
    origin_shot_sequences["chain_goal_time"],
    errors="coerce",
)

# Use nullable integer IDs where missing values are possible.
nullable_int_columns = [
    "anchor_origin_event_id",
    "first_deflection_event_id",
    "first_followup_event_id",
    "chain_goal_event_id",
]

for col in nullable_int_columns:
    origin_shot_sequences[col] = pd.to_numeric(
        origin_shot_sequences[col],
        errors="coerce",
    ).astype("Int64")

In [47]:
final_columns = [
    # Chain identity
    "chain_id",
    "game_id",
    "period",
    "sequence_id",
    "team_id",  # analytical attacking team for the chain
    "raw_first_chance_team_id",
    "n_team_id_overrides_for_chain",

    # Chain timing/counts
    "chain_start_time",
    "chain_end_time",
    "chain_duration_seconds",
    "n_chances_in_chain",

    # Anchor/origin fields
    "anchor_origin_event_id",
    "anchor_origin_period_time",
    "anchor_origin_location",
    "anchor_first_chance_event_id",
    "anchor_first_chance_time",
    "anchor_first_event_type",
    "anchor_first_chance_location",

    # First evaluated chance
    "first_evaluated_chance_event_id",
    "first_evaluated_chance_time",
    "first_evaluated_event_type",
    "first_evaluated_location",
    "first_evaluated_xg",

    # Chain value
    "chain_max_xg",
    "chain_sum_xg",

    # Deflection features
    "has_deflection",
    "n_deflections",
    "first_deflection_event_id",
    "first_deflection_time",
    "deflection_max_xg",
    "deflection_sum_xg",

    # Follow-up features
    "has_followup_chance",
    "n_followup_chances",
    "first_followup_event_id",
    "first_followup_time",
    "followup_max_xg",
    "followup_sum_xg",

    # Goal features
    "chain_goal",
    "chain_goal_event_id",
    "chain_goal_time",
    "chain_goal_player_id",
    "chain_goal_player_name",

    # Diagnostic inherited from 03
    "chain_any_goal_within_2s_flag",
]

origin_shot_sequences_final = origin_shot_sequences[final_columns].copy()

origin_shot_sequences_final.shape


(48673, 43)

In [48]:
origin_shot_sequences_final.head()

,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,...,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg,chain_goal,chain_goal_event_id,chain_goal_time,chain_goal_player_id,chain_goal_player_name,chain_any_goal_within_2s_flag
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,591.73,591.73,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,722.40,722.40,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,736.33,736.33,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,765.73,765.73,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,923.80,923.80,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False


In [49]:
origin_shot_sequences_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 48673 entries, 0 to 48672
Data columns (total 43 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   chain_id                         48673 non-null  str    
 1   game_id                          48673 non-null  str    
 2   period                           48673 non-null  int64  
 3   sequence_id                      48673 non-null  int64  
 4   team_id                          48673 non-null  str    
 5   raw_first_chance_team_id         48673 non-null  str    
 6   n_team_id_overrides_for_chain    48673 non-null  int64  
 7   chain_start_time                 48673 non-null  float64
 8   chain_end_time                   48673 non-null  float64
 9   chain_duration_seconds           48673 non-null  float64
 10  n_chances_in_chain               48673 non-null  int64  
 11  anchor_origin_event_id           48671 non-null  Int64  
 12  anchor_origin_period_time    

In [50]:
# Final uniqueness check.

origin_shot_sequences_final.duplicated("chain_id").sum()

np.int64(0)

## 13. Save Processed Dataset

The processed file is derived data and should not be committed to Git.

The notebook is the reproducible artifact.

In [51]:
output_path = DATA_PROCESSED / "origin_shot_sequences.parquet"

origin_shot_sequences_final.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(origin_shot_sequences_final))

Saved: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed\origin_shot_sequences.parquet
Rows: 48673


In [52]:
pd.read_parquet(output_path).shape

(48673, 43)

## Build Findings

This notebook creates `origin_shot_sequences.parquet`, a chain-level table with one row per non-overlapping shot-created sequence.

Key design decisions:

- Chains are built from evaluated chances, not raw event rows.
- A chance can belong to only one chain.
- Chains are bounded by `game_id`, `period`, `sequence_id`, and analytical attacking `team_id`.
- Defensive deflection rows can be credited to the defending event team, so chain team is corrected to the origin shot team when available.
- Opponent events do not break the chain, but only same-team evaluated chances reset the rolling window.
- The rolling follow-up window is controlled by `FOLLOWUP_WINDOW_SECONDS`.
- `chain_max_xg` is the primary value metric because chances in a flurry are not independent.
- `chain_sum_xg` is retained only as a secondary descriptive field.
- Goals are labeled at the chain level, not treated as direct attribution to every nearby chance.

Next step: add tracking-derived slot-support features at the origin shot moment.
